In [1]:
import tifffile
import os
import sys
import fastplotlib as fpl
import numpy as np
from tqdm import tqdm
import masknmf
import math
import h5py
from natsort import natsorted

import matplotlib.pyplot as plt
%load_ext autoreload

In [2]:
data = tifffile.imread("my_data.tiff") #Shape (num_frames, height, width) 

## Decision 1: Decide whether you want to run piecewise rigid or rigid motion correction

In [4]:


## If rigid, use the following code: 
rigid_motion_correct_config = masknmf.RigidMotionCorrectionConfig(max_shifts = [30, 30]) #Says that max shift in either dimension is 30 pixels

## If nonrigid, try the following recipe. Fill this in with information specific to the recordings you are collecting
fov_height, fov_width = (data.shape[1], data.shape[2]) 
num_blocks = [int(fov_height / 50), int(fov_width / 50)] ## Each piecewise rigid patch has dimensions roughly (50 x 50). If the patches are too small, they become sensitive to noise/low activity
pw_rigid_motion_correct_config = masknmf.PiecewiseRigidMotionCorrectionConfig(max_rigid_shifts=[30, 30],
                                                                     max_deviation_rigid=[1, 1],
                                                                     num_blocks=num_blocks)

# Decision 2: Choose the block size for the PMD compression/denoising method. For axonal data, 32 x 32 is good. If the signals are for some reason very small, then a 20 x 20 blocksize should also be fine. No need to make it smaller

In [5]:
block_sizes = (32, 32)
compress_config = masknmf.CompressDenoiseConfig(block_sizes=(32,32))

## Decision 3: Decide on the correlation thresholds for multipass signal extraction. Either [0.8, 0.8] or [0.8, 0.6] if you want to more aggressively pick up remaining dimmer signals. For most data, 0.8, 0.8 should work, but in either case, we have a SNR-based filter during NMF which removes bad components

In [6]:
conf_list = []
for corr_threshold in [0.8, 0.6]:
    curr_init_conf = masknmf.SuperpixelInitConfig(mad_correlation_threshold=corr_threshold)
    curr_nmf_conf = masknmf.NMFConfig(support_threshold=(0.95, corr_threshold),
                              ring_model_start_pt=0)
    curr_demix_conf = masknmf.SinglepassDemixingConfig(curr_init_conf, curr_nmf_conf)
    conf_list.append(curr_demix_conf)
unfiltered_demixing_config = masknmf.MultipassDemixingConfig(conf_list)

In [2]:

## Specify where the outputs are stored for future reference
outpath_motion_correction = "/some/path/"
outpath_compression = "/some/path/"
outpath_demix = "/some/path/"

## Specify the input data. Here we are loading the data into RAM. If that's not possible, you can pass in a lazily-loaded array that the code will slice in all dimensions
## (time, width, height) to do the processing
results = masknmf.TwoPhotonCalciumPipeline(motion_correct_config=pw_rigid_motion_correct_config,
                                           compress_config=compress_config,
                                           outpath_motion_correction=outpath_motion_correction,
                                           outpath_compression=outpath_compression,
                                           outpath_demixing=outpath_demix,
                                           unfiltered_demixing_config=unfiltered_demixing_config,
                                           frame_batch_size=300)

results.run(data,
            frame_rate = 15, ##Adjust this to your data
            remove_intermediates=False)

## Inspect the demixing results with the single session demixing visualizer

In [3]:
dmr = masknmf.DemixingResults.from_hdf5(outpath_demix) 

In [4]:
demix_vis = masknmf.SingleSessionDemixingVis(dmr, device='cuda')

In [5]:
demix_vis.show()